# Notebook 00 — Cold-tier DuckDB setup

Sets up DuckDB views over the R6 Parquet cold tier and the
live Postgres tables. The other notebooks `%run` this notebook's
setup cell at the top so they don't repeat the view definitions.

When the cold tier is empty (fresh deploy), the views materialise
as empty tables — downstream notebooks display a 'no data' notice
instead of crashing.

_Spec_: `docs/ROUND_13_CALIBRATION_AND_RESEARCH.md` § 3.5


## 1. Imports + DuckDB connection


In [ ]:
import duckdb, os
from pathlib import Path

RESEARCH_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = RESEARCH_DIR / 'duckdb' / 'research.duckdb'
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH))
print(f'DuckDB: {duckdb.__version__} @ {DB_PATH}')


## 2. Register Parquet views over the cold tier

Each table is registered as a view that DuckDB lazily reads
from the partitioned Parquet directory. If a directory is
missing, the view becomes a 0-row table.


In [ ]:
COLD_DIR = RESEARCH_DIR / 'duckdb' / 'cold'
TABLES = ['trades_observed', 'positions_reconstructed', 'leader_strategy_history',
          'multivariate_hawkes_fits', 'follower_pool_state', 'causal_estimates',
          'calibration_loss_history', 'decision_predictions', 'decision_log']

for tbl in TABLES:
    p = COLD_DIR / tbl
    if p.exists():
        con.execute(f"CREATE OR REPLACE VIEW {tbl} AS SELECT * FROM read_parquet('{p}/**/*.parquet')")
        print(f'OK   {tbl} ← {p}')
    else:
        con.execute(f"CREATE OR REPLACE VIEW {tbl} AS SELECT * WHERE 1=0")
        print(f'EMPTY {tbl} (no Parquet at {p})')


## 3. Sanity counts


In [ ]:
for tbl in TABLES:
    n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'{tbl:<35} {n:>12,d} rows')
